In [7]:
# =====================================================
# C-WGAN-GP + Gaussian Penalty (Conditional) with Pretrained cAE
# =====================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# =====================================================
# Paths
# =====================================================
ae_model_path = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\ModelAEV4\cae_fold3.pt")
dataset_path = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1\gcode_layers_V1.npy")
gan_output_dir = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\\ModelGANV1_WGANGP_Gauss")
out_img_dir = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\\OutputGANV1_WGANGP_Gauss")

gan_output_dir.mkdir(parents=True, exist_ok=True)
out_img_dir.mkdir(parents=True, exist_ok=True)

# =====================================================
# Hyperparameters
# =====================================================
latent_size = 512
point_size = 2900
batch_size = 32
epochs = 4001
lr = 3e-5
n_critic = 5
lambda_gp = 10       # gradient penalty coefficient
lambda_gauss = 0.1   # gaussian penalty coefficient
sigma_gauss = 0.05   # gaussian noise std for penalty
device = 'cuda'

# =====================================================
# Load Dataset
# =====================================================
dataset = np.load(dataset_path, allow_pickle=True).item()
points = dataset["data"]
labels = dataset["labels"]
label_map = dataset["label_map"]
areas_norm = dataset["areas_norm"]

num_classes = len(label_map)
cond_dim = 1 + num_classes

flat_points = []
flat_labels = []
flat_areas  = []

for f in range(len(points)):
    for l in range(len(points[f])):
        flat_points.append(points[f][l])      # (P,2)
        flat_labels.append(labels[f])          # scalar
        flat_areas.append(areas_norm[f][l])    # scalar

flat_points = np.array(flat_points, dtype=np.float32)
flat_labels = np.array(flat_labels, dtype=np.int64)
flat_areas  = np.array(flat_areas, dtype=np.float32).reshape(-1, 1)
print(flat_points.shape)   # (?, 2740, 2)
print(flat_labels.shape)   # (?,)
print(flat_areas.shape)    # (?, 1)

class GCodeDataset(Dataset):
    def __init__(self, X, y, a, num_classes):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.a = torch.tensor(a, dtype=torch.float32)
        self.num_classes = num_classes
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx]
        a = self.a[idx]
        y = self.y[idx]
        y1h = F.one_hot(y, num_classes=self.num_classes).float()
        cond = torch.cat([a, y1h], dim=0)
        return x, cond

dataloader = DataLoader(GCodeDataset(flat_points, flat_labels, flat_areas, num_classes),
                        batch_size=batch_size, shuffle=True)

# =====================================================
# Conditional AutoEncoder (Pretrained)
# =====================================================
class PointCloudCAE(nn.Module):
    def __init__(self, point_size, latent_size, cond_dim):
        super().__init__()
        self.P = point_size
        self.enc = nn.Sequential(
            nn.Linear(point_size * 2, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, latent_size),
        )
        self.dec = nn.Sequential(
            nn.Linear(latent_size + cond_dim, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 1024), nn.ReLU(),
            nn.Linear(1024, point_size * 2),
        )
    def forward(self, x, cond):
        B = x.shape[0]
        x_flat = x.view(B, -1)
        z = self.enc(x_flat)
        zc = torch.cat([z, cond], dim=1)
        out = self.dec(zc).view(B, self.P, 2)
        return out

cae = PointCloudCAE(point_size, latent_size, cond_dim).to(device)
cae.load_state_dict(torch.load(ae_model_path, map_location=device))
cae.eval()
for p in cae.parameters():
    p.requires_grad = False
# =====================================================
# Generator + Critic
#note1: (g512-1024 works good with c1024-256 )
# =====================================================
class Generator(nn.Module):
    def __init__(self, latent_dim, cond_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 512),
            nn.BatchNorm1d(512), nn.LeakyReLU(0.2),

            nn.Linear(512, 512),
            nn.BatchNorm1d(512), nn.LeakyReLU(0.2),

            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024), nn.LeakyReLU(0.2),

            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048), nn.LeakyReLU(0.2),
            
            nn.Linear(2048, latent_dim)   # latent vector output
        )
    def forward(self, z, cond):
        return self.model(torch.cat([z, cond], dim=1))
        
class Critic(nn.Module):
    def __init__(self, latent_dim, cond_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 2048),
            nn.LeakyReLU(0.2),

            nn.Linear(2048, 1024),
            nn.LeakyReLU(0.2),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 1)   # no sigmoid for WGAN
        )
    def forward(self, z, cond):
        return self.model(torch.cat([z, cond], dim=1))

G = Generator(latent_size, cond_dim).to(device)
D = Critic(latent_size, cond_dim).to(device)

opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.0, 0.9))

# =====================================================
# Regularization Functions
# =====================================================
def compute_gradient_penalty(D, real_samples, fake_samples, cond):
    alpha = torch.rand(real_samples.size(0), 1).to(device)
    alpha = alpha.expand_as(real_samples)

    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_interpolates = D(interpolates, cond)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    gradient_norm = gradients.norm(2, dim=1)
    gp = lambda_gp * ((gradient_norm - 1) ** 2).mean()
    return gp

def compute_gaussian_penalty(D, real_samples, fake_samples, cond, sigma=0.05):
    noise_real = real_samples + sigma * torch.randn_like(real_samples)
    noise_fake = fake_samples + sigma * torch.randn_like(fake_samples)
    d_real_noisy = D(noise_real, cond)
    d_fake_noisy = D(noise_fake, cond)
    penalty = 0.5 * (d_real_noisy ** 2).mean() + 0.5 * (d_fake_noisy ** 2).mean()
    return penalty

# =====================================================
# Training Loop
# =====================================================
g_losses, d_losses = [], []
for epoch in range(epochs):
    for i, (x, cond) in enumerate(dataloader):
        x, cond = x.to(device), cond.to(device)

        # --- Real latent from CAE ---
        with torch.no_grad():
            z_real = cae.enc(x.view(x.size(0), -1))

        # --- Fake latent ---
        z_noise = torch.randn(x.size(0), latent_size).to(device)
        z_fake = G(z_noise, cond)

        # --- Critic Update ---
        d_real = D(z_real, cond)
        d_fake = D(z_fake.detach(), cond)
        gp = compute_gradient_penalty(D, z_real, z_fake, cond)
        gauss_pen = compute_gaussian_penalty(D, z_real, z_fake, cond, sigma_gauss)

        d_loss = -(torch.mean(d_real) - torch.mean(d_fake)) + gp + lambda_gauss * gauss_pen

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # --- Generator Update (every n_critic steps) ---
        if i % n_critic == 0:
            z_noise = torch.randn(x.size(0), latent_size).to(device)  # NEW
            z_fake = G(z_noise, cond)                                  # NEW
        
            g_fake = D(z_fake, cond)
            g_loss = -torch.mean(g_fake)
        
            opt_G.zero_grad()
            g_loss.backward()
            opt_G.step()
            

    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    print(f"Epoch {epoch+1}/{epochs} | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

    # Save and visualize every 100 epochs
    if (epoch+1) % 500 == 0:
        torch.save(G.state_dict(), gan_output_dir / f"wgangp_gauss_generator_epoch{epoch+1}.pt")
        torch.save(D.state_dict(), gan_output_dir / f"wgangp_gauss_critic_epoch{epoch+1}.pt")
    if (epoch+1) % 25 == 0:
        with torch.no_grad():
            z_noise = torch.randn(num_classes, latent_size).to(device)
            labels_eye = torch.eye(num_classes).to(device)
            areas_dummy = torch.rand(num_classes, 1).to(device)
            conds = torch.cat([areas_dummy, labels_eye], dim=1)

            fake_latents = G(z_noise, conds)
            decoded = cae.dec(torch.cat([fake_latents, conds], dim=1)).view(
                num_classes, point_size, 2
            ).cpu().numpy()

            fig, axs = plt.subplots(1, num_classes, figsize=(20, 4))
            for i in range(num_classes):
                axs[i].scatter(decoded[i][:, 0], decoded[i][:, 1], s=1)
                axs[i].set_title(f"Class {i} | area={areas_dummy[i].item():.2f}")
                axs[i].axis("equal")
                axs[i].grid(True)
            plt.tight_layout()
            plt.savefig(out_img_dir / f"epoch_{epoch+1:04d}_samples.png")
            plt.close()

    # === Save loss curve each epoch ===
    plt.figure(figsize=(10,5))
    plt.plot(g_losses, label="Generator Loss")
    plt.plot(d_losses, label="Critic Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"C-WGAN-GP+Gaussian Training Loss (up to epoch {epoch+1})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_img_dir /"img"/ f"epoch_{epoch+1:04d}_loss.png")
    plt.close()

# Final save
torch.save(G.state_dict(), gan_output_dir / "wgangp_gauss_generator_final.pt")
torch.save(D.state_dict(), gan_output_dir / "wgangp_gauss_critic_final.pt")


(5900, 2900, 2)
(5900,)
(5900, 1)
Epoch 1/4001 | D Loss: -3.6374 | G Loss: 5.0727
Epoch 2/4001 | D Loss: -4.6481 | G Loss: 4.2057
Epoch 3/4001 | D Loss: -4.8422 | G Loss: 3.3431
Epoch 4/4001 | D Loss: -4.9256 | G Loss: 2.6550
Epoch 5/4001 | D Loss: -4.7371 | G Loss: 2.3270
Epoch 6/4001 | D Loss: -4.5243 | G Loss: 2.5192
Epoch 7/4001 | D Loss: -4.3653 | G Loss: 2.1177
Epoch 8/4001 | D Loss: -4.2871 | G Loss: 2.0326
Epoch 9/4001 | D Loss: -4.3156 | G Loss: 1.9899
Epoch 10/4001 | D Loss: -4.2689 | G Loss: 1.7992
Epoch 11/4001 | D Loss: -4.0728 | G Loss: 1.6506
Epoch 12/4001 | D Loss: -4.0488 | G Loss: 1.7124
Epoch 13/4001 | D Loss: -3.9682 | G Loss: 1.6124
Epoch 14/4001 | D Loss: -3.7957 | G Loss: 1.3870
Epoch 15/4001 | D Loss: -3.7451 | G Loss: 1.3930
Epoch 16/4001 | D Loss: -3.7059 | G Loss: 1.4090
Epoch 17/4001 | D Loss: -3.6899 | G Loss: 1.4427
Epoch 18/4001 | D Loss: -3.6706 | G Loss: 1.3187
Epoch 19/4001 | D Loss: -3.5413 | G Loss: 1.2032
Epoch 20/4001 | D Loss: -3.4851 | G Loss: 1.